# Lab 2.2: Escalation & Ambiguity Resolution

**What you'll build:** A customer support agent with explicit escalation triggers — and learn why sentiment-based escalation fails in both directions.

**Estimated time:** 20-25 minutes

| Phase | What happens | Time |
|-------|-------------|------|
| 1 | The Wrong Way — sentiment-based escalation mis-routes easy and hard cases | 5 min |
| 2 | The Right Way — explicit trigger conditions produce correct escalation decisions | 5 min |
| 3 | Your Turn — write escalation logic for a set of support scenarios | 10 min |
| 4 | Stress Test — verify your triggers are consistent across runs | 5 min |

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

## The Setup

You're designing the escalation logic for a customer support agent. The agent needs to decide when to handle issues itself and when to hand off to a human.

The challenge: sentiment analysis sounds like a good escalation signal, but it fails in predictable ways:
- **Angry customer + simple issue** = over-escalation (wastes human time)
- **Calm customer + complex issue** = under-escalation (poor outcome)

We'll test both approaches against realistic scenarios.

In [ ]:
# Test scenarios with known correct escalation decisions
SCENARIOS = [
    {
        "id": "S1",
        "message": "This is absolutely ridiculous! I've been waiting 3 days for a simple order status update. Your service is terrible!",
        "context": {"issue_type": "order_status", "order_exists": True, "status_available": True},
        "correct_action": "handle",
        "reason": "Angry but simple — order status is available, agent can resolve immediately"
    },
    {
        "id": "S2",
        "message": "Hi, I have a question about my bill. There seems to be a charge I don't recognize for a 'Premium Data Recovery Service' at $2,499. I didn't sign up for this.",
        "context": {"issue_type": "unauthorized_charge", "amount": 2499, "in_policy": False},
        "correct_action": "escalate",
        "reason": "Calm but complex — unauthorized charge exceeds agent authority, policy gap"
    },
    {
        "id": "S3",
        "message": "I'd like to speak to a manager, please.",
        "context": {"issue_type": "human_request", "explicit_request": True},
        "correct_action": "escalate",
        "reason": "Customer explicitly requests human — escalate immediately, don't investigate first"
    },
    {
        "id": "S4",
        "message": "I'm so frustrated!!! My package shows delivered but I never got it. This is the WORST experience ever!!!",
        "context": {"issue_type": "missing_package", "tracking_shows_delivered": True, "replacement_policy": True},
        "correct_action": "handle",
        "reason": "Very angry but resolvable — replacement policy exists, agent can offer solution"
    },
    {
        "id": "S5",
        "message": "Hello, I've been a customer for 12 years and was wondering if there's any loyalty program or discount available for long-term customers like me?",
        "context": {"issue_type": "loyalty_discount", "policy_exists": False, "customer_tenure_years": 12},
        "correct_action": "escalate",
        "reason": "Calm and polite but policy gap — no loyalty discount policy, agent cannot make one up"
    }
]

print(f"Loaded {len(SCENARIOS)} test scenarios")
for s in SCENARIOS:
    print(f"  {s['id']}: {s['correct_action'].upper()} — {s['reason'][:60]}...")

---

## Phase 1: The Wrong Approach

Sentiment-based escalation: analyze the customer's emotional state and escalate when sentiment is negative. Sounds reasonable — let's see it fail.

In [ ]:
SENTIMENT_PROMPT = """You are a customer support escalation system. Analyze the customer's message
and determine if it should be escalated to a human agent.

ESCALATION RULE: If the customer's sentiment is negative or frustrated, escalate.
If the customer seems calm and satisfied, handle it with the AI agent.

Customer message: {message}

Respond with ONLY a JSON object:
{{"action": "escalate" or "handle", "reasoning": "brief explanation"}}"""


def run_escalation(prompt_template, scenarios):
    """Run escalation logic against all scenarios."""
    results = []
    for s in scenarios:
        prompt = prompt_template.format(message=s["message"], context=json.dumps(s["context"]))
        response = client.messages.create(
            model=MODEL,
            max_tokens=200,
            messages=[{"role": "user", "content": prompt}]
        )
        raw = response.content[0].text.strip()
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
        try:
            result = json.loads(raw)
        except json.JSONDecodeError:
            result = {"action": "unknown", "reasoning": raw}
        
        result["scenario_id"] = s["id"]
        result["correct"] = s["correct_action"]
        result["is_correct"] = result.get("action") == s["correct_action"]
        results.append(result)
    return results


sentiment_results = run_escalation(SENTIMENT_PROMPT, SCENARIOS)

print("=== SENTIMENT-BASED ESCALATION RESULTS ===")
for r in sentiment_results:
    correct_tag = "CORRECT" if r["is_correct"] else "WRONG"
    print(f"\n  {r['scenario_id']}: {r.get('action', '?').upper()} (should be {r['correct'].upper()}) — {correct_tag}")
    print(f"    Reasoning: {r.get('reasoning', 'N/A')}")

accuracy = sum(1 for r in sentiment_results if r["is_correct"]) / len(sentiment_results)
print(f"\nSentiment-based accuracy: {accuracy:.0%}")

### Why did that fail?

- **Over-escalation:** Angry customers with simple issues (S1, S4) get escalated when the agent could resolve their issue in seconds.
- **Under-escalation:** Calm customers with complex issues (S2, S5) don't trigger sentiment-based escalation, even though their issues require human authority.
- **Sentiment is orthogonal to escalation need.** The customer's emotional state tells you how they *feel*, not whether the *issue* requires human intervention.
- **The correct escalation signal is the issue type and agent capability**, not the customer's tone.

---

## Phase 2: The Right Approach

Explicit trigger-based escalation: define concrete conditions that warrant escalation, independent of sentiment.

In [ ]:
TRIGGER_PROMPT = """You are a customer support escalation system. Analyze the customer's message
and context to determine if it should be escalated to a human agent.

ESCALATION TRIGGERS — escalate if ANY of these conditions are true:
1. Customer explicitly requests to speak to a human, manager, or supervisor
   (escalate IMMEDIATELY — do NOT investigate or try to resolve first)
2. Policy gap: the customer's request is not covered by any existing policy
   (e.g., no policy for the type of discount requested)
3. The issue exceeds agent authority (e.g., unauthorized charges, amounts above $500)
4. Three or more resolution attempts have failed

DO NOT ESCALATE based on:
- Customer sentiment or tone alone (angry customers may have simple issues)
- The agent's self-assessed confidence
- The length or complexity of the customer's message

HANDLE (do not escalate) when:
- The issue is within agent capability AND policy exists
- Even if the customer is frustrated, offer resolution first

Customer message: {message}
Context: {context}

Respond with ONLY a JSON object:
{{"action": "escalate" or "handle", "trigger": "which trigger matched or why handle", "reasoning": "brief explanation"}}"""


trigger_results = run_escalation(TRIGGER_PROMPT, SCENARIOS)

print("=== TRIGGER-BASED ESCALATION RESULTS ===")
for r in trigger_results:
    correct_tag = "CORRECT" if r["is_correct"] else "WRONG"
    print(f"\n  {r['scenario_id']}: {r.get('action', '?').upper()} (should be {r['correct'].upper()}) — {correct_tag}")
    print(f"    Trigger: {r.get('trigger', 'N/A')}")
    print(f"    Reasoning: {r.get('reasoning', 'N/A')}")

trigger_accuracy = sum(1 for r in trigger_results if r["is_correct"]) / len(trigger_results)
print(f"\nTrigger-based accuracy: {trigger_accuracy:.0%}")

In [ ]:
# Side-by-side comparison
print("=" * 65)
print("COMPARISON: SENTIMENT vs TRIGGER-BASED")
print("=" * 65)
print(f"{'Scenario':<8} {'Correct':<12} {'Sentiment':<12} {'Trigger':<12}")
print(f"{'-'*8} {'-'*12} {'-'*12} {'-'*12}")
for s_r, t_r in zip(sentiment_results, trigger_results):
    s_mark = "OK" if s_r["is_correct"] else "WRONG"
    t_mark = "OK" if t_r["is_correct"] else "WRONG"
    print(f"{s_r['scenario_id']:<8} {s_r['correct']:<12} {s_mark:<12} {t_mark:<12}")

print(f"\n{'Accuracy':<8} {'':12} {accuracy:<11.0%} {trigger_accuracy:<11.0%}")

---

## Phase 3: Your Turn

Write escalation trigger logic for a new set of scenarios. Your triggers must handle all cases correctly.

**Your goal:** 100% accuracy on all scenarios — correct escalation or handling for each.

In [ ]:
CHALLENGE_SCENARIOS = [
    {
        "id": "C1",
        "message": "I want a refund AND a replacement AND a discount on my next order. This has been a nightmare.",
        "context": {"issue_type": "refund_request", "refund_policy": True, "refund_amount": 89.99, "replacement_available": True, "additional_discount_policy": False},
        "correct_action": "escalate",
        "reason": "Refund and replacement are in policy, but additional discount is a policy gap"
    },
    {
        "id": "C2",
        "message": "WORST COMPANY EVER!!! I WILL NEVER BUY FROM YOU AGAIN!!! My order arrived one day late.",
        "context": {"issue_type": "late_delivery", "delay_days": 1, "compensation_policy": True, "shipping_credit_available": True},
        "correct_action": "handle",
        "reason": "Extremely angry but simple — 1-day delay with compensation policy available"
    },
    {
        "id": "C3",
        "message": "Could you please transfer me to your billing department? I'd rather discuss this with them directly.",
        "context": {"issue_type": "billing_question", "explicit_transfer_request": True},
        "correct_action": "escalate",
        "reason": "Explicit request for human department — honor immediately"
    },
    {
        "id": "C4",
        "message": "Hi there! Just wondering when my order will arrive? The tracking hasn't updated in a while.",
        "context": {"issue_type": "order_tracking", "tracking_available": True, "estimated_delivery": "2024-12-08"},
        "correct_action": "handle",
        "reason": "Friendly and simple — tracking info is available, agent can provide it"
    },
    {
        "id": "C5",
        "message": "I need help with my subscription. I've been double-charged for 6 months and the total overcharge is $714. I'd like this resolved.",
        "context": {"issue_type": "billing_error", "overcharge_amount": 714, "months_affected": 6, "refund_authority_limit": 500},
        "correct_action": "escalate",
        "reason": "Calm but amount ($714) exceeds agent refund authority ($500)"
    }
]

print(f"Loaded {len(CHALLENGE_SCENARIOS)} challenge scenarios")
for s in CHALLENGE_SCENARIOS:
    print(f"  {s['id']}: {s['correct_action'].upper()} — {s['reason'][:60]}...")

In [ ]:
# =============================================================
# TODO: Write your escalation trigger prompt
# =============================================================
#
# Requirements:
# - Use explicit trigger conditions (not sentiment)
# - Must handle: explicit human requests, policy gaps, authority limits
# - Must NOT over-escalate angry-but-simple cases
#
# Think about:
# - How do you express the agent's authority limit ($500)?
# - How do you detect policy gaps vs policy-covered requests?
# - How do you prevent sentiment from influencing the decision?

YOUR_ESCALATION_PROMPT = """You are a customer support escalation system.

# TODO: Replace with your explicit trigger conditions.
# Define concrete escalation triggers and handle conditions.

Customer message: {message}
Context: {context}

Respond with ONLY a JSON object:
{{"action": "escalate" or "handle", "trigger": "which trigger matched", "reasoning": "brief explanation"}}"""


your_results = run_escalation(YOUR_ESCALATION_PROMPT, CHALLENGE_SCENARIOS)

print("=== YOUR ESCALATION RESULTS ===")
for r in your_results:
    correct_tag = "CORRECT" if r["is_correct"] else "WRONG"
    print(f"\n  {r['scenario_id']}: {r.get('action', '?').upper()} (should be {r['correct'].upper()}) — {correct_tag}")
    print(f"    Trigger: {r.get('trigger', 'N/A')}")

In [ ]:
# =============================================================
# Checker: validates your solution
# =============================================================

your_accuracy = sum(1 for r in your_results if r["is_correct"]) / len(your_results)

print("=== YOUR SCORECARD ===")
print(f"  Accuracy: {your_accuracy:.0%}")
print()

wrong = [r for r in your_results if not r["is_correct"]]
if your_accuracy == 1.0:
    print("  PASSED — 100% accuracy!")
    print("  Your trigger conditions correctly handled all cases.")
else:
    print(f"  {len(wrong)} incorrect decision(s):")
    for r in wrong:
        scenario = next(s for s in CHALLENGE_SCENARIOS if s["id"] == r["scenario_id"])
        print(f"    {r['scenario_id']}: said {r.get('action', '?')}, should be {r['correct']}")
        print(f"      Reason: {scenario['reason']}")
    print("\n  Revise your trigger conditions and try again.")

---

## Phase 4: Stress Test

Run your triggers multiple times to ensure consistency.

In [ ]:
print("Running your triggers 5 times against all scenarios...\n")

run_accuracies = []
for run in range(5):
    results = run_escalation(YOUR_ESCALATION_PROMPT, CHALLENGE_SCENARIOS)
    acc = sum(1 for r in results if r["is_correct"]) / len(results)
    run_accuracies.append(acc)
    wrong_ids = [r["scenario_id"] for r in results if not r["is_correct"]]
    print(f"  Run {run+1}: {acc:.0%}", end="")
    if wrong_ids:
        print(f" (wrong: {wrong_ids})")
    else:
        print(" (perfect)")

print(f"\n=== CONSISTENCY ===")
print(f"Accuracies: {[f'{a:.0%}' for a in run_accuracies]}")
if all(a == 1.0 for a in run_accuracies):
    print("Perfect consistency — your triggers produce correct decisions every time.")
else:
    print("Inconsistent — some scenarios flip between runs. Tighten your trigger conditions.")

### Key Takeaways

1. **Sentiment is not an escalation signal.** Angry customers may have trivial issues; calm customers may have complex ones.
2. **Explicit triggers are deterministic.** Customer requests human, policy gap, authority exceeded, attempts exhausted — these are observable, testable conditions.
3. **Customer preference overrides everything.** When someone says "talk to a human," escalate immediately. Don't investigate first.
4. **Policy gaps require escalation.** The agent cannot invent policy — only a human with authority can make exceptions.
5. **Authority limits must be concrete.** "Amounts over $500" is testable; "large amounts" is not.